# 03 — Paula: notebook reorganizado para instalação e execução sequencial

**Primeiro objetivo: carregar DINOv3 e medir o piloto.** Extração completa, classificação e ajuste parcial continuam desativados nas configurações atuais.

1. Abra esta versão em um kernel novo. Preserve resultados que ainda não foram salvos antes de encerrar a sessão anterior.
2. Execute a primeira célula de instalação. Se terminar com **ETAPA 0 OK**, continue **sem reiniciar**.
3. Se a célula informar bibliotecas antigas na memória, reinicie somente o kernel e volte à primeira célula. Ela é idempotente: não reinstala versões já corretas.
4. Execute as demais células em ordem. Pare no primeiro erro; não tente compensá-lo executando células posteriores.
5. Não acrescente instalações no meio do notebook nem use `importlib.reload` para trocar versões de Transformers.

O protocolo observado no arquivo enviado está em `/kaggle/input/datasets/paulaeduardadelima/datasets-not02`. A configuração foi corrigida para essa pasta, com descoberta alternativa caso o caminho mude.

A versão enviada já mostrava validação de manifesto/listas e métricas sintéticas concluídas. O carregamento do DINOv3 falhou por versão incorreta e, depois disso, o benchmark falhou porque `adapter` não existia. **Nenhum benchmark real é declarado concluído aqui.** As saídas antigas foram removidas para não parecerem resultados desta execução.

Este notebook mantém os módulos e a metodologia da versão enviada. Não foi executado na sua sessão Kaggle a partir daqui. Instalação/importações dependem do ambiente e o piloto real deve confirmar a configuração.


**Execução por etapas; sem resultados pré-preenchidos.** Importe no Kaggle, anexe o OCT-C8 e a pasta extraída do ZIP do notebook 02 (incluindo `manifest_used.csv` e `data/splits/registry.json`). Use uma GPU para piloto/extração/ajuste; classificação linear pode usar CPU. Execute um backbone por sessão.

Ordem: configurar → validar dados/listas → métricas → carregar backbone → medir batch em 256 imagens de treino → extrair uma vez → selecionar classificadores na validação → registrar seleção → avaliação final opcional. Ajuste parcial é outra seção, começando com uma época de 10% para medir custo.

CNN simples, RETFound e ViT supervisionado são de Ana Júlia; este notebook fornece o avaliador, o formato de embeddings e a rotina linear reutilizável para ela. Não implementa esses três treinamentos.

**Datas:** as janelas 21–23/10 do plano são históricas. Registrar as datas reais das execuções, sem declarar tarefas retroativamente concluídas.

Os arquivos locais de `/kaggle/working` não sobrevivem necessariamente à sessão. Preserve saídas e restaure checkpoints/embeddings antes de retomar. O notebook não publica arquivos nem agenda sessões automaticamente.

**Verificação desta entrega:** sintaxe das células e módulos conferida; testes locais passaram para listas, retomada/checksums dos embeddings, alinhamento de IDs e scaler só no subconjunto. A retomada do treinamento foi comparada a uma execução ininterrupta em um backbone pequeno sintético em CPU, com resultado idêntico. Os checkpoints reais DINOv3/ResNet e a execução CUDA não foram testados neste ambiente; o benchmark no Kaggle é necessário. Não há métricas reais do OCT nesta entrega.

## 0. Instalar antes de importar

A célula usa somente a biblioteca padrão. Fixa Transformers4.56.2, Hub0.36.2 e Safetensors0.8.0; mantém PyTorch/CUDA/Pillow existentes. Remove Gradio/Diffusers, que não são usados e já haviam sido removidos na tentativa enviada. As versões efetivamente usadas são registradas depois.

Uma mensagem de reinício aqui é deliberada quando há módulos antigos na memória. Se a preparação terminar em OK, **não reinicie**: siga para 0B.

In [2]:
# PRIMEIRA CÉLULA DE CÓDIGO — não importar Transformers antes dela.
import sys
import subprocess
import importlib.metadata

BOOTSTRAP_READY = False
ENV_READY = False
MODEL_READY = False
EXPECTED_PACKAGES = {
    'transformers': '4.56.2',
    'huggingface_hub': '0.36.2',
    'safetensors': '0.8.0',
}
# Gradio e Diffusers não fazem parte deste projeto e conflitam com o Hub fixado.
REMOVE_UNUSED_CONFLICTS = True

def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

# Detecta módulos já carregados, sem importá-los para fazer a verificação.
tracked = ['transformers', 'huggingface_hub', 'tokenizers', 'safetensors']
loaded_before = {name: getattr(sys.modules[name], '__version__', None)
                 for name in tracked if name in sys.modules}
needs_install = any(installed_version(k) != v for k, v in EXPECTED_PACKAGES.items())

if REMOVE_UNUSED_CONFLICTS:
    unused = [p for p in ['gradio', 'diffusers'] if installed_version(p) is not None]
    if unused:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', *unused], check=True)

if needs_install:
    print('Instalando as versões do projeto. PyTorch/CUDA não serão reinstalados.')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
         *[f'{name}=={v}' for name, v in EXPECTED_PACKAGES.items()]],
        check=True,
    )
else:
    print('Versões corretas já instaladas; nenhuma reinstalação necessária.')

for name, expected in EXPECTED_PACKAGES.items():
    actual = installed_version(name)
    print(f'{name}: instalada={actual}')
    if actual != expected:
        raise RuntimeError(f'A instalação de {name} não terminou com a versão esperada.')

stale = [name for name, loaded in loaded_before.items()
         if loaded != installed_version(name)]
if (needs_install and loaded_before) or stale:
    raise RuntimeError(
        'INSTALAÇÃO CONCLUÍDA, MAS O KERNEL JÁ TINHA BIBLIOTECAS CARREGADAS. '
        'Reinicie somente o kernel e execute novamente desde esta primeira célula. '
        'Se o ambiente inteiro for recriado, esta célula instalará o necessário novamente. '
        'Não execute as próximas células antes do reinício.'
    )
BOOTSTRAP_READY = True
print('ETAPA 0 OK — prossiga sem reiniciar o kernel.')


Instalando as versões do projeto. PyTorch/CUDA não serão reinstalados.


  Using cached transformers-4.56.2-py3-none-any.whl.metadata (40 kB)
Using cached transformers-4.56.2-py3-none-any.whl (11.6 MB)


transformers: instalada=4.56.2
huggingface_hub: instalada=0.36.2
safetensors: instalada=0.8.0
ETAPA 0 OK — prossiga sem reiniciar o kernel.


## 0B. Importar uma única vez e verificar o ambiente

In [3]:
assert globals().get('BOOTSTRAP_READY', False), 'Execute a primeira célula e siga sua mensagem.'
ENV_READY = False
from pathlib import Path
import json, hashlib, time, gc, shutil, platform, datetime
import numpy as np
import pandas as pd
import PIL
import torch
import torchvision
import transformers
import huggingface_hub
import safetensors
from transformers import AutoImageProcessor, AutoModel
from IPython.display import display

for name, module in [('transformers', transformers), ('huggingface_hub', huggingface_hub), ('safetensors', safetensors)]:
    disk = importlib.metadata.version(name)
    memory = module.__version__
    print(f'{name}: instalada={disk} | carregada={memory}')
    if disk != EXPECTED_PACKAGES[name] or memory != disk:
        raise RuntimeError('Versões divergentes: volte à primeira célula; reinicie o kernel se solicitado.')

print('Pillow:', PIL.__version__)
print('PyTorch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA:', torch.version.cuda, '| GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
ENV_READY = True
print('ETAPA 0B OK — importações concluídas.')


transformers: instalada=4.56.2 | carregada=4.56.2
huggingface_hub: instalada=0.36.2 | carregada=0.36.2
safetensors: instalada=0.8.0 | carregada=0.8.0
Pillow: 13.0.0
PyTorch: 2.11.0+cu128 | torchvision: 0.26.0+cu128
CUDA: 12.8 | GPU disponível: True
GPU: Tesla T4
ETAPA 0B OK — importações concluídas.


## 1. Caminhos e decisões de execução

`PROTOCOL_DIR` é a pasta que contém `manifest_used.csv`, e não a pasta das imagens. Não use `octc8_manifest_audit_v1.csv` como seleção. Se tiver somente o ZIP do notebook02, extraia-o primeiro para uma pasta de trabalho.

Os controles `DATA_APPROVED` e `BUDGET_REVIEWED` registram decisões reais da dupla; não são permissões de acesso. O piloto pode usar o candidato. Execuções principais exigem seleção de dados e protocolo definidos para que os resultados sejam comparáveis.

Nesta versão `LOAD_BACKBONE=True` e `RUN_BENCHMARK=True` já estão definidos para o piloto; as etapas principais continuam desativadas. O código não usa tokens em arquivos de configuração.

In [31]:
assert globals().get('ENV_READY', False), 'Execute as células 0 e 0B primeiro.'
PROTOCOL_DIR = Path('/kaggle/input/datasets/paulaeduardadelima/datasets-not02')
DATA_ROOT = Path('/kaggle/input/datasets/obulisainaren/retinal-oct-c8')
OUT_ROOT = Path('/kaggle/working/oct_models')
CACHE = Path('/kaggle/temp/oct_models_cache')
BACKBONE = 'dinov3_vitl16'  # depois: 'resnet50'
# BACKBONE = 'resnet50'

# Mesmo commit do teste inicial. Trocar implica nova identidade de features.
DINO_REVISION = 'ea8dc2863c51be0a264bab82070e3e8836b02d51'
DATA_APPROVED = True
APPROVED_MANIFEST_SHA256 = (
    "80d6eedc883410b47beeee6b1c72e72d69309ba38ac2595fc8f2590e6928103d"
)
REVIEWERS = "Paula Eduarda de Lima"
BUDGET_REVIEWED = True




# O piloto de ajuste parcial carrega seu próprio modelo.
LOAD_BACKBONE = False
RUN_BENCHMARK = False
RUN_EXTRACTION = False
RUN_LINEAR = False
RUN_LINEAR_FINAL_TEST = False


# Executar somente uma época para medir o custo.
RUN_FT_EPOCH_PILOT = True
RUN_PARTIAL_FT = False
RUN_FT_FINAL_TEST = False



EXTRACTION_BATCH = 8
CHUNK_SIZE = 256
MAX_NEW_CHUNKS = None


PREVIOUS_OUTPUT_DIR = ""


BATCH_CANDIDATES = [2,4,8,16,32]
EXTRACTION_BATCH = 16  # substituído pela recomendação se executar benchmark
CHUNK_SIZE = 256
MAX_NEW_CHUNKS = None  # exemplo: 2 para testar interrupção/retomada
PREVIOUS_OUTPUT_DIR = ''  # pasta salva anteriormente que contém features/, runs/ etc.
LINEAR_C = [.001,.01,.1,1.,10.]
LINEAR_MAX_ITER = 1000
# Começar com [10] e [17]; completar os demais após medir custo.
FT_PERCENTAGES = [10]
FT_SEEDS = [17]
FT_LRS = [1e-5,3e-5]
FT_EPOCHS = 15
FT_PATIENCE = 5
FT_BATCH = 2
FT_ACCUMULATION = 8
FT_EVAL_BATCH = 8


# Valida a pasta sem confundir manifesto auditado com o protocolo completo.
def is_protocol_folder(path):
    return path.is_dir() and (path/'manifest_used.csv').is_file() and (path/'data/splits/registry.json').is_file()

if not is_protocol_folder(PROTOCOL_DIR):
    candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('manifest_used.csv')
                         if is_protocol_folder(p.parent)})
    if len(candidates) != 1:
        print('Pastas completas encontradas:', candidates)
        raise RuntimeError('Anexe o pacote completo do notebook 02 e ajuste PROTOCOL_DIR para sua pasta.')
    PROTOCOL_DIR = candidates[0]
assert DATA_ROOT.is_dir(), 'Anexe o OCT-C8 e confira DATA_ROOT.'
if LOAD_BACKBONE or RUN_FT_EPOCH_PILOT or RUN_PARTIAL_FT or RUN_FT_FINAL_TEST:
    assert torch.cuda.is_available(), 'Ative GPU. Se o ambiente reiniciar, execute desde a primeira célula.'
print('Protocolo:', PROTOCOL_DIR)
print('Imagens:', DATA_ROOT)
print('Execução atual: backbone=', BACKBONE, '; benchmark=', RUN_BENCHMARK)


Protocolo: /kaggle/input/datasets/paulaeduardadelima/datasets-not02
Imagens: /kaggle/input/datasets/obulisainaren/retinal-oct-c8
Execução atual: backbone= dinov3_vitl16 ; benchmark= False


## 2. Exportar módulos compartilhados

`oct_protocol.py` mantém a definição de métricas do notebook02. `oct_experiments.py` acrescenta verificação das listas, extração em blocos e regressão logística. `oct_gpu.py` implementa adaptadores e ajuste parcial.

Os fontes ficam embutidos para o notebook ser autocontido; a célula os grava para importar e commitar. Alterações no código mudam o identificador da execução, evitando reutilização silenciosa.

In [32]:
assert globals().get('ENV_READY', False), 'Execute a preparação do ambiente.'
SOURCES = {'oct_protocol.py': "import numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score\n\nCLASSES = ['AMD','CNV','CSR','DME','DR','DRUSEN','MH','NORMAL']\n\ndef make_subsets(manifest, seeds=(17,42,2026), percentages=(1,5,10,25,100)):\n    train=manifest.loc[manifest.split=='train'].copy()\n    result={}\n    for seed in seeds:\n        permutations={}\n        for cls in range(8):\n            rows=train.loc[train.class_id==cls].sort_values('image_id').reset_index(drop=True)\n            rng=np.random.default_rng(np.random.SeedSequence([seed,cls]))\n            permutations[cls]=rows.iloc[rng.permutation(len(rows))]\n        previous=set()\n        for pct in percentages:\n            parts=[rows.iloc[:max(1,len(rows)*pct//100)] for rows in permutations.values()]\n            subset=pd.concat(parts).sort_values('image_id').reset_index(drop=True)\n            ids=set(subset.image_id)\n            assert previous.issubset(ids)\n            previous=ids\n            result[(seed,pct)]=subset\n        assert previous==set(train.image_id)\n    return result\n\ndef prepare_rgb(path):\n    # Imagem inteira; distorce a proporção original, sem recorte/padding.\n    with Image.open(path) as im:\n        return im.convert('RGB').resize((224,224),Image.Resampling.BICUBIC)\n\ndef normalize_rgb(image, mean, std):\n    mean=np.asarray(mean,dtype=np.float32); std=np.asarray(std,dtype=np.float32)\n    if mean.shape!=(3,) or std.shape!=(3,) or not np.isfinite(mean).all() or not np.isfinite(std).all() or (std<=0).any():\n        raise ValueError('Informe mean/std verificados para os três canais.')\n    x=np.asarray(image,dtype=np.float32)/255.0\n    if x.shape!=(224,224,3): raise ValueError('Esperado RGB 224x224.')\n    return ((x-mean)/std).transpose(2,0,1).copy()\n\ndef evaluate_predictions(reference, predictions):\n    columns=['p_'+c for c in CLASSES]\n    if not {'image_id','class_id'}.issubset(reference.columns): raise ValueError('Referência inválida.')\n    if not {'image_id',*columns}.issubset(predictions.columns): raise ValueError('Faltam probabilidades por classe.')\n    if not reference.image_id.is_unique or not predictions.image_id.is_unique: raise ValueError('IDs duplicados.')\n    if len(reference)==0 or set(reference.image_id)!=set(predictions.image_id): raise ValueError('IDs não correspondem.')\n    labels=reference.class_id.to_numpy()\n    if not np.isin(labels,np.arange(8)).all(): raise ValueError('Rótulos fora de 0..7.')\n    aligned=reference[['image_id','class_id']].merge(predictions[['image_id']+columns],on='image_id',validate='one_to_one')\n    y=aligned.class_id.to_numpy(dtype=int); p=aligned[columns].to_numpy(dtype=float)\n    if not np.isfinite(p).all() or (p<0).any() or (p>1).any() or not np.allclose(p.sum(1),1,atol=1e-5):\n        raise ValueError('Esperadas probabilidades finitas, entre 0 e 1, somando 1; não logits.')\n    pred=p.argmax(1)\n    cm=confusion_matrix(y,pred,labels=np.arange(8)); support=cm.sum(1)\n    recall=[float(cm[i,i]/support[i]) if support[i] else None for i in range(8)]\n    auc=[float(roc_auc_score((y==i).astype(int),p[:,i])) if 0<(y==i).sum()<len(y) else None for i in range(8)]\n    acc=float(accuracy_score(y,pred)); ba=float(np.mean([r for r in recall if r is not None]))\n    balanced=bool((support>0).all() and (support==support[0]).all())\n    if balanced and not np.isclose(acc,ba): raise AssertionError('Erro em métricas de conjunto balanceado.')\n    metrics={'n':len(y),'class_order':CLASSES,'macro_f1':float(f1_score(y,pred,labels=np.arange(8),average='macro',zero_division=0)),\n             'accuracy':acc,'balanced_accuracy':ba,'balanced_accuracy_scope':'classes_present',\n             'recall_per_class':dict(zip(CLASSES,recall)),'support':dict(zip(CLASSES,map(int,support))),\n             'auroc_per_class':dict(zip(CLASSES,auc)),\n             'auroc_macro':float(np.mean(auc)) if all(a is not None for a in auc) else None,\n             'auroc_macro_defined_classes':float(np.mean([a for a in auc if a is not None])) if any(a is not None for a in auc) else None,\n             'auroc_classes_defined':sum(a is not None for a in auc),\n             'balanced_support':balanced,'accuracy_minus_balanced_accuracy':acc-ba}\n    normalized=np.divide(cm,support[:,None],out=np.zeros_like(cm,dtype=float),where=support[:,None]!=0)\n    return metrics,pd.DataFrame(cm,index=CLASSES,columns=CLASSES),pd.DataFrame(normalized,index=CLASSES,columns=CLASSES)\n", 'oct_experiments.py': '"""Shared CPU utilities: strict identities, resumable embeddings and linear selection."""\nfrom pathlib import Path\nimport hashlib,json,time,warnings,os\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.exceptions import ConvergenceWarning\nfrom oct_protocol import CLASSES,evaluate_predictions\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(1024*1024),b\'\'): h.update(block)\n    return h.hexdigest()\ndef digest(obj): return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,allow_nan=False).encode()).hexdigest()\ndef atomic_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\');tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\ndef read_csv(path): return pd.read_csv(path,dtype={\'image_id\':str},keep_default_na=False)\ndef bind_config(folder,config):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);p=folder/\'config.json\'\n    if p.exists():\n        if json.loads(p.read_text())!=config: raise ValueError(\'Configuração mudou: use nova pasta.\')\n    else: atomic_json(p,config)\n    return folder\n\ndef verify_lists(protocol_dir):\n    root=Path(protocol_dir); manifest=read_csv(root/\'manifest_used.csv\')\n    h=sha(root/\'manifest_used.csv\'); registry=json.loads((root/\'data/splits/registry.json\').read_text())\n    assert registry[\'manifest_sha256\']==h\n    assert len(manifest)==23850 and manifest.image_id.is_unique and manifest.relative_path.is_unique\n    assert manifest.groupby(\'split\').size().to_dict()=={\'test\':2797,\'train\':18260,\'val\':2793}\n    assert manifest.class_name.map(dict(zip(CLASSES,range(8)))).eq(manifest.class_id).all()\n    assert manifest.read_ok.astype(str).str.lower().eq(\'true\').all()\n    for c in [\'file_sha256\',\'pixel_sha256\']:\n        assert manifest[c].str.fullmatch(\'[0-9a-f]{64}\').all() and manifest[c].is_unique\n    lookup=manifest.set_index(\'image_id\'); result={}\n    cols=[\'relative_path\',\'class_name\',\'class_id\',\'split\',\'file_sha256\',\'pixel_sha256\']\n    for key,item in registry[\'subsets\'].items():\n        p=root/\'data/splits\'/item[\'file\'];assert sha(p)==item[\'sha256\'],str(p)\n        df=read_csv(p);assert df.image_id.is_unique\n        pd.testing.assert_frame_equal(df.set_index(\'image_id\')[cols].sort_index(),lookup.loc[df.image_id,cols].sort_index())\n        assert set(df.split)=={key if key in [\'val\',\'test\'] else \'train\'}\n        result[key]=df\n    for split in [\'val\',\'test\']: assert set(result[split].image_id)==set(manifest.loc[manifest.split==split,\'image_id\'])\n    for seed in [17,42,2026]:\n        previous=set()\n        for pct in [1,5,10,25,100]:\n            df=result[f\'p{pct:02d}_seed{seed}\']; ids=set(df.image_id)\n            assert previous<=ids;previous=ids\n            for cls in range(8):\n                n=((manifest.split==\'train\') & (manifest.class_id==cls)).sum()\n                assert (df.class_id==cls).sum()==max(1,int(n)*pct//100)\n        assert previous==set(manifest.loc[manifest.split==\'train\',\'image_id\'])\n    return manifest,result,h\n\ndef validate_chunk(path,rows,feature_id,dim):\n    with np.load(path,allow_pickle=False) as z:\n        assert z[\'feature_id\'].item()==feature_id\n        assert z[\'image_id\'].tolist()==rows.image_id.tolist()\n        assert z[\'class_id\'].tolist()==rows.class_id.tolist()\n        assert z[\'split\'].tolist()==rows.split.tolist()\n        x=z[\'embeddings\'];assert x.shape==(len(rows),dim) and x.dtype==np.float32 and np.isfinite(x).all()\n        return x.copy()\n\ndef extract_chunks(frame,folder,config,embed,chunk_size=256,max_new_chunks=None):\n    folder=bind_config(folder,config);fid=digest(config); dim=config[\'dimension\']\n    frame=frame.sort_values([\'split\',\'image_id\']).reset_index(drop=True)\n    start=time.perf_counter();new=reused=0\n    for split,part in frame.groupby(\'split\',sort=True):\n        part=part.reset_index(drop=True); sub=folder/split;sub.mkdir(exist_ok=True)\n        for a in range(0,len(part),chunk_size):\n            rows=part.iloc[a:a+chunk_size];path=sub/f\'chunk_{a//chunk_size:05d}.npz\'\n            meta=path.with_suffix(\'.json\')\n            if path.exists():\n                if meta.exists(): assert json.loads(meta.read_text())[\'sha256\']==sha(path),\'Bloco alterado.\'\n                validate_chunk(path,rows,fid,dim);reused+=1\n            else:\n                if max_new_chunks is not None and new>=max_new_chunks:\n                    return {\'complete\':False,\'new\':new,\'reused\':reused,\'seconds\':time.perf_counter()-start}\n                x=embed(rows).astype(np.float32)\n                assert x.shape==(len(rows),dim) and np.isfinite(x).all()\n                tmp=path.with_suffix(\'.tmp\')\n                with open(tmp,\'wb\') as f: np.savez_compressed(f,embeddings=x,image_id=rows.image_id.to_numpy(dtype=str),class_id=rows.class_id.to_numpy(dtype=np.int64),split=rows.split.to_numpy(dtype=str),feature_id=np.array(fid))\n                tmp.replace(path);validate_chunk(path,rows,fid,dim);new+=1\n            if not meta.exists(): atomic_json(meta,{\'sha256\':sha(path),\'n\':len(rows),\'feature_id\':fid})\n            atomic_json(folder/\'progress.json\',{\'last_chunk\':str(path.relative_to(folder)),\'new_this_call\':new,\'reused_this_call\':reused})\n    report={\'complete\':True,\'new\':new,\'reused\':reused,\'seconds\':time.perf_counter()-start,\'feature_id\':fid}\n    atomic_json(folder/\'complete.json\',report);return report\n\ndef load_split(folder,reference,split):\n    folder=Path(folder); cfg=json.loads((folder/\'config.json\').read_text());fid=digest(cfg)\n    expected=reference.loc[reference.split==split].sort_values(\'image_id\').reset_index(drop=True)\n    assert len(expected)>0\n    files=sorted((folder/split).glob(\'chunk_*.npz\'));expected_n=(len(expected)+cfg[\'chunk_size\']-1)//cfg[\'chunk_size\']\n    assert len(files)==expected_n, \'Extração incompleta ou blocos extras.\'\n    blocks=[]\n    for i,p in enumerate(files):\n        assert p.name==f\'chunk_{i:05d}.npz\'\n        assert json.loads(p.with_suffix(\'.json\').read_text())[\'sha256\']==sha(p)\n        blocks.append(validate_chunk(p,expected.iloc[i*cfg[\'chunk_size\']:(i+1)*cfg[\'chunk_size\']],fid,cfg[\'dimension\']))\n    return expected,np.concatenate(blocks)\n\ndef predict_frame(clf,X,ids):\n    assert np.array_equal(clf.classes_,np.arange(8))\n    p=pd.DataFrame(clf.predict_proba(X),columns=[\'p_\'+c for c in CLASSES]);p.insert(0,\'image_id\',list(ids));return p\n\ndef save_evaluation(folder,reference,predictions,prefix):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    m,cm,norm=evaluate_predictions(reference,predictions)\n    atomic_json(folder/f\'{prefix}_metrics.json\',m)\n    predictions.to_csv(folder/f\'{prefix}_predictions.csv\',index=False)\n    cm.to_csv(folder/f\'{prefix}_confusion_counts.csv\');norm.to_csv(folder/f\'{prefix}_confusion_normalized.csv\')\n    return m\n\ndef fit_linear(train_ref,Xtrain,val_ref,Xval,subset,folder,config):\n    assert set(train_ref.split)=={\'train\'} and set(val_ref.split)=={\'val\'} and set(subset.split)=={\'train\'}\n    assert set(train_ref.image_id).isdisjoint(val_ref.image_id)\n    assert set(subset.image_id)<=set(train_ref.image_id) and subset.image_id.is_unique\n    ids=subset.sort_values(\'image_id\').image_id.tolist();positions=pd.Series(np.arange(len(train_ref)),index=train_ref.image_id)\n    selected=positions.loc[ids].to_numpy(); X=Xtrain[selected];y=train_ref.iloc[selected].class_id.to_numpy()\n    folder=bind_config(folder,config)\n    if (folder/\'selected.json\').exists():\n        result=json.loads((folder/\'selected.json\').read_text());assert sha(folder/\'classifier.joblib\')==result[\'classifier_sha256\'];return result\n    trials=[];best=None\n    for C in sorted(config[\'C_grid\']):\n        start=time.perf_counter();clf=make_pipeline(StandardScaler(),LogisticRegression(C=C,solver=\'lbfgs\',max_iter=config[\'max_iter\'],random_state=17))\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter(\'always\');clf.fit(X,y)\n        messages=[str(w.message) for w in caught]\n        valid=not any(issubclass(w.category,ConvergenceWarning) for w in caught)\n        record={\'C\':C,\'valid\':valid,\'warnings\':messages,\'seconds\':time.perf_counter()-start}\n        if valid:\n            assert np.allclose(clf[0].mean_,X.mean(0),rtol=1e-4,atol=1e-5),\'Scaler não usa só o subconjunto.\'\n            pred=predict_frame(clf,Xval,val_ref.image_id);m,_,_=evaluate_predictions(val_ref,pred)\n            record[\'validation_macro_f1\']=m[\'macro_f1\']\n            if best is None or m[\'macro_f1\']>best[0]: best=(m[\'macro_f1\'],C,clf,pred)\n        trials.append(record);atomic_json(folder/\'trials.json\',trials)\n    if best is None: raise RuntimeError(\'Nenhum candidato convergiu. Revisar orçamento igualmente para todos.\')\n    score,C,clf,pred=best;tmp=folder/\'classifier.tmp\';joblib.dump(clf,tmp);tmp.replace(folder/\'classifier.joblib\')\n    m=save_evaluation(folder,val_ref,pred,\'validation\')\n    chosen={\'C\':C,\'validation_macro_f1\':score,\'classifier_sha256\':sha(folder/\'classifier.joblib\'),\'config_hash\':digest(config),\'test_used_for_selection\':False}\n    atomic_json(folder/\'selected.json\',chosen);return chosen\n', 'oct_gpu.py': '"""GPU adapters and epoch-boundary partial fine-tuning."""\nfrom pathlib import Path\nimport time,gc,json,random,math\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom PIL import Image\nfrom oct_protocol import CLASSES,evaluate_predictions\nfrom oct_experiments import sha,digest,atomic_json,bind_config,save_evaluation\n\ndef seed_all(seed):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n\ndef sync():\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n\ndef image_tensor(row,root,mean,std):\n    p=Path(root)/row.relative_path\n    assert sha(p)==row.file_sha256, f\'Imagem diferente da auditada: {row.image_id}\'\n    with Image.open(p) as im: arr=np.array(im.convert(\'RGB\').resize((224,224),Image.Resampling.BICUBIC),dtype=np.float32)/255.\n    arr=(arr-np.asarray(mean,dtype=np.float32))/np.asarray(std,dtype=np.float32)\n    return torch.from_numpy(arr.transpose(2,0,1).copy())\n\nclass RowsDataset(torch.utils.data.Dataset):\n    def __init__(self,rows,root,mean,std): self.rows=rows.reset_index(drop=True);self.root=root;self.mean=mean;self.std=std\n    def __len__(self):return len(self.rows)\n    def __getitem__(self,i):\n        row=self.rows.iloc[i]\n        return image_tensor(row,self.root,self.mean,self.std),int(row.class_id),str(row.image_id)\n\ndef loader(rows,root,mean,std,batch,shuffle=False,generator=None):\n    return torch.utils.data.DataLoader(RowsDataset(rows,root,mean,std),batch_size=batch,shuffle=shuffle,num_workers=0,pin_memory=torch.cuda.is_available(),generator=generator)\n\nclass Backbone:\n    def __init__(self,name,device,cache,revision,token=None):\n        self.name=name;self.device=torch.device(device); self.cache=Path(cache);self.cache.mkdir(parents=True,exist_ok=True)\n        if name==\'dinov3_vitl16\':\n            import transformers\n            from transformers import AutoImageProcessor,AutoModel\n            assert transformers.__version__==\'4.56.2\',\'Reinicie o kernel após instalar a versão fixada.\'\n            assert revision and len(revision)==40,\'Fixe o commit DINOv3.\'\n            repo=\'facebook/dinov3-vitl16-pretrain-lvd1689m\'\n            proc=AutoImageProcessor.from_pretrained(repo,revision=revision,token=token,cache_dir=str(cache),use_fast=True)\n            self.mean=list(proc.image_mean);self.std=list(proc.image_std)\n            assert proc.do_normalize and proc.do_rescale and np.isclose(proc.rescale_factor,1/255)\n            self.model=AutoModel.from_pretrained(repo,revision=revision,token=token,cache_dir=str(cache),torch_dtype=torch.float32,attn_implementation=\'sdpa\')\n            self.dim=int(self.model.config.hidden_size);assert self.dim==1024\n            self.identity={\'repo\':repo,\'revision\':revision,\'feature\':\'last_hidden_state[:,0,:] after final norm\',\'processor\':proc.to_dict()}\n        elif name==\'resnet50\':\n            from torchvision.models import resnet50,ResNet50_Weights\n            from urllib.parse import urlparse\n            torch.hub.set_dir(str(cache));w=ResNet50_Weights.IMAGENET1K_V2\n            self.model=resnet50(weights=w);self.model.fc=nn.Identity();self.dim=2048\n            t=w.transforms();self.mean=list(t.mean);self.std=list(t.std)\n            weight_path=Path(torch.hub.get_dir())/\'checkpoints\'/Path(urlparse(w.url).path).name\n            self.identity={\'weights\':\'ResNet50_Weights.IMAGENET1K_V2\',\'url\':w.url,\'weights_sha256\':sha(weight_path),\'feature\':\'global_avgpool_flatten_before_fc\',\'geometry_override\':\'whole-image bicubic224 instead of official resize232 + center crop224\'}\n        else: raise ValueError(name)\n        self.identity=json.loads(json.dumps(self.identity))\n        self.model.eval().requires_grad_(False).to(self.device)\n        self.preprocessing={\'rgb\':True,\'size\':[224,224],\'resize\':\'PIL_BICUBIC_whole_image\',\'crop\':False,\'aspect_ratio_preserved\':False,\'mean\':self.mean,\'std\':self.std,\'rescale\':1/255,\'augmentations\':False}\n    def forward(self,x):\n        return self.model(pixel_values=x).last_hidden_state[:,0,:] if self.name==\'dinov3_vitl16\' else self.model(x)\n    def embed(self,rows,root,batch=8):\n        outputs=[];ids=[];labels=[]\n        self.model.eval()\n        with torch.inference_mode():\n            for x,y,batch_ids in loader(rows,root,self.mean,self.std,batch):\n                with torch.autocast(device_type=self.device.type,dtype=torch.float16,enabled=self.device.type==\'cuda\'):\n                    z=self.forward(x.to(self.device,non_blocking=True))\n                outputs.append(z.float().cpu().numpy());ids.extend(batch_ids);labels.extend(y.tolist())\n        assert ids==rows.image_id.tolist() and labels==rows.class_id.tolist()\n        result=np.concatenate(outputs);assert result.shape==(len(rows),self.dim) and np.isfinite(result).all()\n        return result\n\ndef benchmark(adapter,rows,root,batches,total_images,folder):\n    assert set(rows.split)=={\'train\'}\n    report=[];device=adapter.device\n    for batch in batches:\n        try:\n            adapter.embed(rows.iloc[:min(batch,len(rows))],root,batch);sync()\n            if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()\n            sync();start=time.perf_counter();z=adapter.embed(rows,root,batch);sync();seconds=time.perf_counter()-start\n            # Measure storage separately, to include it in the extrapolation.\n            folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);temp=folder/\'benchmark_write.npz\'\n            s=time.perf_counter();np.savez_compressed(temp,embeddings=z);write_seconds=time.perf_counter()-s;temp.unlink()\n            allocated=torch.cuda.max_memory_allocated()/2**30 if device.type==\'cuda\' else None\n            reserved=torch.cuda.max_memory_reserved()/2**30 if device.type==\'cuda\' else None\n            total=(seconds+write_seconds)*total_images/len(rows)\n            report.append({\'batch\':batch,\'ok\':True,\'n\':len(rows),\'seconds\':seconds,\'write_seconds\':write_seconds,\'images_per_second\':len(rows)/seconds,\'peak_allocated_gib\':allocated,\'peak_reserved_gib\':reserved,\'estimated_extraction_seconds\':total,\'budget_with_margin_seconds\':total*1.3+600})\n        except torch.cuda.OutOfMemoryError:\n            gc.collect();torch.cuda.empty_cache();report.append({\'batch\':batch,\'ok\':False,\'reason\':\'CUDA OOM\'})\n        atomic_json(Path(folder)/\'benchmark.json\',report)\n    successful=[r for r in report if r[\'ok\']]\n    if device.type==\'cuda\':\n        capacity=torch.cuda.get_device_properties(device).total_memory/2**30\n        safe=[r for r in successful if r[\'peak_reserved_gib\']<=.8*capacity]\n    else:safe=successful\n    if not safe:raise RuntimeError(\'Sem batch com margem de 20%; testar batches menores.\')\n    best=max(safe,key=lambda r:r[\'images_per_second\'])\n    atomic_json(Path(folder)/\'batch_choice.json\',best);return report,best[\'batch\']\n\nclass PartialDINO(nn.Module):\n    def __init__(self,backbone):\n        super().__init__();self.backbone=backbone\n        assert hasattr(backbone,\'layer\') and len(backbone.layer)>=2,\'Arquitetura inesperada para transformers4.56.2.\'\n        self.backbone.requires_grad_(False)\n        for block in self.backbone.layer[-2:]:block.requires_grad_(True)\n        self.head=nn.Linear(backbone.config.hidden_size,8)\n    def train(self,mode=True):\n        super().train(mode);self.backbone.eval()\n        for block in self.backbone.layer[-2:]:block.train(mode)\n        self.head.train(mode);return self\n    def forward(self,x):return self.head(self.backbone(pixel_values=x).last_hidden_state[:,0,:])\n\ndef trainable_state(net):return {n:p.detach().cpu().clone() for n,p in net.named_parameters() if p.requires_grad}\ndef restore_trainable(net,state):\n    expected={n for n,p in net.named_parameters() if p.requires_grad};assert set(state)==expected\n    with torch.no_grad():\n        for n,p in net.named_parameters():\n            if n in state:p.copy_(state[n].to(p.device))\ndef atomic_torch(path,obj):\n    path=Path(path);tmp=path.with_suffix(\'.tmp\');torch.save(obj,tmp);tmp.replace(path)\ndef optimizer_to(optimizer,device):\n    for state in optimizer.state.values():\n        for k,v in state.items():\n            if torch.is_tensor(v):state[k]=v.to(device)\n\ndef probabilities(net,rows,adapter,root,batch):\n    net.eval();result=[];ids=[];labels=[]\n    with torch.no_grad():\n        for x,y,bid in loader(rows,root,adapter.mean,adapter.std,batch):\n            with torch.autocast(device_type=adapter.device.type,dtype=torch.float16,enabled=adapter.device.type==\'cuda\'):\n                logits=net(x.to(adapter.device))\n            result.append(torch.softmax(logits.float(),1).cpu().numpy());ids.extend(bid);labels.extend(y.tolist())\n    assert ids==rows.image_id.tolist() and labels==rows.class_id.tolist()\n    frame=pd.DataFrame(np.concatenate(result),columns=[\'p_\'+c for c in CLASSES]);frame.insert(0,\'image_id\',ids);return frame\n\ndef fit_partial(adapter,train,val,root,folder,config):\n    assert set(train.split)=={\'train\'} and set(val.split)=={\'val\'} and set(train.image_id).isdisjoint(val.image_id)\n    seed_all(config[\'seed\']);net=PartialDINO(adapter.model).to(adapter.device)\n    folder=bind_config(folder,config);path=folder/\'last.pt\'\n    if (folder/\'result.json\').exists():\n        result=json.loads((folder/\'result.json\').read_text())\n        assert sha(folder/\'best.pt\')==result[\'best_sha256\']\n        return result\n    atomic_json(folder/\'parameters.json\',{\'trainable\':sum(p.numel() for p in net.parameters() if p.requires_grad),\'total\':sum(p.numel() for p in net.parameters()),\'names\':[n for n,p in net.named_parameters() if p.requires_grad]})\n    optimizer=torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],lr=config[\'lr\'],weight_decay=config[\'weight_decay\'])\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=adapter.device.type==\'cuda\')\n    generator=torch.Generator().manual_seed(config[\'seed\'])\n    start_epoch=0;history=[];best_score=-1.;best_state=None;best_epoch=-1;bad_epochs=0;done=False\n    if path.exists():\n        # Somente checkpoints criados por este projeto; torch pickle é confiado aqui.\n        state=torch.load(path,map_location=\'cpu\',weights_only=False)\n        assert state[\'config\']==config\n        restore_trainable(net,state[\'trainable\']);optimizer.load_state_dict(state[\'optimizer\']);optimizer_to(optimizer,adapter.device);scaler.load_state_dict(state[\'scaler\'])\n        torch.set_rng_state(state[\'torch_rng\']);np.random.set_state(state[\'numpy_rng\']);random.setstate(state[\'python_rng\'])\n        if adapter.device.type==\'cuda\':torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n        generator.set_state(state[\'loader_rng\']);start_epoch=state[\'next_epoch\'];history=state[\'history\'];best_score=state[\'best_score\'];best_state=state[\'best_state\'];best_epoch=state[\'best_epoch\'];bad_epochs=state[\'bad_epochs\'];done=state[\'done\']\n    for epoch in range(start_epoch,config[\'epochs\']):\n        if done:break\n        trainloader=loader(train,root,adapter.mean,adapter.std,config[\'batch\'],True,generator)\n        net.train();optimizer.zero_grad(set_to_none=True);loss_sum=0.;group_n=0\n        if adapter.device.type==\'cuda\':torch.cuda.reset_peak_memory_stats()\n        sync();t=time.perf_counter()\n        for step,(x,y,_) in enumerate(trainloader):\n            x=x.to(adapter.device);y=y.to(adapter.device)\n            with torch.autocast(device_type=adapter.device.type,dtype=torch.float16,enabled=adapter.device.type==\'cuda\'):\n                loss=nn.functional.cross_entropy(net(x),y,reduction=\'sum\')\n            if not torch.isfinite(loss):raise FloatingPointError(\'Loss não finita; retome do checkpoint anterior após investigar.\')\n            scaler.scale(loss).backward();group_n+=len(y);loss_sum+=float(loss.detach())\n            if (step+1)%config[\'accumulation\']==0 or step+1==len(trainloader):\n                scaler.unscale_(optimizer)\n                for p in net.parameters():\n                    if p.grad is not None:p.grad.div_(group_n)\n                torch.nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad],1.)\n                scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True);group_n=0\n        sync();train_seconds=time.perf_counter()-t;t=time.perf_counter()\n        pred=probabilities(net,val,adapter,root,config[\'eval_batch\']);sync();val_seconds=time.perf_counter()-t\n        metrics,_,_=evaluate_predictions(val,pred);score=metrics[\'macro_f1\']\n        improved=score>best_score\n        if improved:best_score=score;best_state=trainable_state(net);best_epoch=epoch;bad_epochs=0\n        else:bad_epochs+=1\n        done=bad_epochs>=config[\'patience\'] or epoch+1==config[\'epochs\']\n        history.append({\'epoch\':epoch+1,\'loss\':loss_sum/len(train),\'val_macro_f1\':score,\'train_seconds\':train_seconds,\'validation_seconds\':val_seconds,\'peak_allocated_gib\':torch.cuda.max_memory_allocated()/2**30 if adapter.device.type==\'cuda\' else None,\'improved\':improved})\n        state={\'config\':config,\'trainable\':trainable_state(net),\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'next_epoch\':epoch+1,\'history\':history,\'best_score\':best_score,\'best_state\':best_state,\'best_epoch\':best_epoch,\'bad_epochs\':bad_epochs,\'done\':done,\'torch_rng\':torch.get_rng_state(),\'cuda_rng\':torch.cuda.get_rng_state_all() if adapter.device.type==\'cuda\' else [],\'numpy_rng\':np.random.get_state(),\'python_rng\':random.getstate(),\'loader_rng\':generator.get_state()}\n        s=time.perf_counter();atomic_torch(path,state);save_seconds=time.perf_counter()-s\n        history[-1][\'checkpoint_seconds\']=save_seconds\n        atomic_json(folder/\'history.json\',history)\n        print(history[-1])\n    assert best_state is not None\n    restore_trainable(net,best_state)\n    pred=probabilities(net,val,adapter,root,config[\'eval_batch\']);metrics=save_evaluation(folder,val,pred,\'validation\')\n    atomic_torch(folder/\'best.pt\',{\'config\':config,\'trainable\':best_state,\'best_epoch\':best_epoch+1})\n    result={\'best_epoch\':best_epoch+1,\'validation_macro_f1\':metrics[\'macro_f1\'],\'epochs_completed\':len(history),\'limit_reached\':len(history)>=config[\'epochs\'],\'improved_at_limit\':bool(history[-1][\'improved\'] and len(history)>=config[\'epochs\']),\'best_sha256\':sha(folder/\'best.pt\')}\n    atomic_json(folder/\'result.json\',result);return result\n'}
SRC=OUT_ROOT/'src';SRC.mkdir(parents=True,exist_ok=True)
for name,source in SOURCES.items(): (SRC/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(SRC))
# Recarrega os módulos desta versão ao reexecutar.
for name in ['oct_protocol','oct_experiments','oct_gpu']: sys.modules.pop(name,None)
from oct_protocol import CLASSES,evaluate_predictions
from oct_experiments import *
CODE_HASH=digest(SOURCES)

assert callable(verify_lists)
print('Módulos exportados e importados com sucesso.')


Módulos exportados e importados com sucesso.


## 3. Validar dados e listas, registrar protocolo

As listas de frações são lidas do notebook02 e não sorteadas novamente. O código verifica hashes, IDs, classes, divisões, aninhamento e contagens. Validação: 2.793 rótulos; teste: 2.797. As frações reduzem apenas o treino.

Orçamento linear: cinco valores de C, até 1000 iterações, mesmo solver e critério em todos os backbones. Empates favorecem menor C. Falhas de convergência são registradas e excluídas da seleção; ampliar o orçamento exige nova versão e a mesma oportunidade para os modelos.

Ajuste parcial: duas taxas de aprendizado, 15 épocas, paciência 5, AdamW, weight decay 0,01. Ana Júlia deve adotar orçamento comparável para RETFound. Registrar tempos e falhas, pois configurações iguais não significam custo igual.

In [33]:
assert callable(globals().get('verify_lists')), 'Execute a seção 2: Exportar módulos compartilhados.'
manifest,lists,MANIFEST_HASH=verify_lists(PROTOCOL_DIR)
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','Pillow','scikit-learn','joblib']}
for p in ['torch','torchvision','transformers','huggingface_hub','safetensors']:
    try:versions[p]=importlib.metadata.version(p)
    except importlib.metadata.PackageNotFoundError:pass
ENV={'python':platform.python_version(),'packages':versions}
if DATA_APPROVED:
    assert APPROVED_MANIFEST_SHA256==MANIFEST_HASH and REVIEWERS.strip()

def require_main():
    assert DATA_APPROVED and APPROVED_MANIFEST_SHA256==MANIFEST_HASH and REVIEWERS.strip(), 'Registre o manifesto revisado antes dos experimentos principais.'
    assert BUDGET_REVIEWED, 'Registre a revisão conjunta do orçamento.'

OUT=OUT_ROOT/(MANIFEST_HASH[:12]+'_'+CODE_HASH[:8]);OUT.mkdir(parents=True,exist_ok=True)
if PREVIOUS_OUTPUT_DIR:
    prev=Path(PREVIOUS_OUTPUT_DIR)
    assert prev.is_dir() and prev.resolve()!=OUT.resolve()
    for source in prev.rglob('*'):
        if source.is_file():
            dest=OUT/source.relative_to(prev)
            if dest.exists():assert sha(dest)==sha(source),'Conflito com arquivo da sessão atual.'
            else:dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(source,dest)
atomic_json(OUT/'environment.json',ENV)
PROTOCOL={'manifest_sha256':MANIFEST_HASH,'code_hash':CODE_HASH,'class_order':CLASSES,
 'data_approved':DATA_APPROVED,'reviewers':REVIEWERS,'budget_reviewed':BUDGET_REVIEWED,
 'linear_C':LINEAR_C,'linear_max_iter':LINEAR_MAX_ITER,'primary_metric':'macro_f1',
 'selection_split':'val','test_selection':False,'standard_scaler_fit':'selected_train_only',
 'ft_lrs':FT_LRS,'ft_epochs':FT_EPOCHS,'ft_patience':FT_PATIENCE,'ft_weight_decay':.01,
 'limitations':['patient_separation_unverified','pretraining_independence_unverified','near_duplicates_not_excluded','architecture_pretraining_scale_objective_confounded','whole_image_resize_distorts_aspect_ratio']}
atomic_json(OUT/'protocol.json',PROTOCOL)
print('Manifesto:',MANIFEST_HASH,'Saídas:',OUT)
display(manifest.groupby(['class_name','split']).size().unstack().reindex(CLASSES))


Manifesto: 80d6eedc883410b47beeee6b1c72e72d69309ba38ac2595fc8f2590e6928103d Saídas: /kaggle/working/oct_models/80d6eedc8834_a305b2c7


split,test,train,val
class_name,,,
AMD,350,2300,350
CNV,350,2257,347
CSR,350,2300,350
DME,349,2272,349
DR,350,2300,350
DRUSEN,349,2233,348
MH,350,2300,350
NORMAL,349,2298,349


## 4. Testar métricas e contratos de avaliação

AUROC usa probabilidades na ordem fixa das oito classes; não rótulos previstos. Classes sem positivos/negativos recebem null. Macro-F1 é principal. O teste auditado não é perfeitamente balanceado (349/350), portanto não exigir igualdade entre accuracy e balanced accuracy. A igualdade é verificada num caso sintético balanceado.

Não use números dos testes sintéticos como resultado científico.

In [34]:
reference=pd.DataFrame({'image_id':[f'test_{i}' for i in range(16)],'class_id':np.tile(np.arange(8),2)})
pred=pd.DataFrame(np.eye(8)[reference.class_id],columns=['p_'+c for c in CLASSES]);pred.insert(0,'image_id',reference.image_id)
m,_,_=evaluate_predictions(reference,pred.sample(frac=1,random_state=17))
assert m['macro_f1']==m['accuracy']==m['balanced_accuracy']==m['auroc_macro']==1
bad=pred.copy();bad.loc[0,'image_id']='wrong'
try:evaluate_predictions(reference,bad)
except ValueError:pass
else:raise AssertionError('IDs inválidos aceitos.')
missing,_,_=evaluate_predictions(reference.iloc[:1],pred.iloc[:1])
assert missing['auroc_macro'] is None
atomic_json(OUT/'metrics_self_test.json',{'passed':True,'synthetic':True})
print('Métricas verificadas. Ordem:',CLASSES)


Métricas verificadas. Ordem: ['AMD', 'CNV', 'CSR', 'DME', 'DR', 'DRUSEN', 'MH', 'NORMAL']


## 5. Carregar um backbone

DINOv3: CLS final, 1.024 dimensões. ResNet-50 IMAGENET1K_V2: saída do global average pooling antes da FC, 2.048 dimensões. Ambos: eval(), pesos fixos, RGB e resize integral bicúbico224; extração AMP float16 e armazenamento float32.

A normalização vem do processador DINOv3 ou da configuração dos pesos ResNet. A geometria escolhida difere do resize232+crop224 oficial do ResNet V2; essa adaptação comum evita cortar regiões OCT e está registrada. Não atribuir diferenças apenas ao domínio de pré-treinamento.

DINOv3 exige acesso aprovado e `HF_TOKEN` nos Secrets do Kaggle. Não imprimir ou salvar o token. A configuração não precisa de tokens para ResNet.

In [8]:
MODEL_READY = False
if LOAD_BACKBONE:
    import torch
    from oct_gpu import Backbone,benchmark,seed_all
    assert torch.cuda.is_available(),'Ative GPU para piloto/extração.'
    assert BACKBONE in ['dinov3_vitl16','resnet50']
    if 'adapter' in globals(): del adapter;gc.collect();torch.cuda.empty_cache()
    token=None
    if BACKBONE=='dinov3_vitl16':
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_TOKEN')
    seed_all(17)
    adapter=Backbone(BACKBONE,'cuda:0',CACHE,DINO_REVISION,token)
    token=None
    MODEL_READY = True
    GPU={'name':torch.cuda.get_device_name(0),'total_gib':torch.cuda.get_device_properties(0).total_memory/2**30}
    print(BACKBONE,'dimensão:',adapter.dim,'GPU:',GPU)
    atomic_json(OUT/f'{BACKBONE}_model.json',{'identity':adapter.identity,'preprocessing':adapter.preprocessing,'gpu':GPU})
else:
    print('Carregamento desativado. Para o primeiro piloto: LOAD_BACKBONE=True e RUN_BENCHMARK=True.')


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /kaggle/temp/oct_models_cache/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 194MB/s]


resnet50 dimensão: 2048 GPU: {'name': 'Tesla T4', 'total_gib': 14.56219482421875}


## 6. Benchmark de 256 imagens e escolha de batch

Amostra: 32 imagens por classe, somente treino, seed17. Medição após warm-up inclui leitura, conferência dos hashes, processamento e inferência; escrita é medida separadamente. Memória é pico alocado/reservado pelo PyTorch na GPU, não RAM do host.

Testa batches 2/4/8/16/32; OOM é registrado. Escolhe o mais rápido entre os que deixaram pelo menos 20% de memória reservada livre. A estimativa para 23.850 imagens acrescenta 30% e 10 minutos de margem; download dos pesos, exportação e variações de I/O podem exigir mais. Use esses números para planejar sessões, não como garantia de duração.

In [9]:
if RUN_BENCHMARK:
    assert LOAD_BACKBONE and globals().get('MODEL_READY', False), 'O backbone não carregou. Resolva a seção 5 antes do benchmark.'
    pilot=pd.concat([manifest.loc[(manifest.split=='train')&(manifest.class_id==i)].sort_values('image_id').sample(n=32,random_state=17) for i in range(8)]).sort_values('image_id').reset_index(drop=True)
    report,EXTRACTION_BATCH=benchmark(adapter,pilot,DATA_ROOT,BATCH_CANDIDATES,len(manifest),OUT/'pilots'/BACKBONE)
    display(pd.DataFrame(report));print('Batch recomendado:',EXTRACTION_BATCH)
    # Reprocessa uma imagem isolada: checa representação e alinhamento no piloto.
    batch_vectors=adapter.embed(pilot.iloc[:4],DATA_ROOT,EXTRACTION_BATCH)
    single=adapter.embed(pilot.iloc[:1],DATA_ROOT,1)[0]
    cosine=float(np.dot(single,batch_vectors[0])/(np.linalg.norm(single)*np.linalg.norm(batch_vectors[0])))
    assert cosine>.999,'Diferença inesperada: investigar processamento/batching.'
    atomic_json(OUT/'pilots'/BACKBONE/'alignment.json',{'passed':True,'first_image_id':pilot.iloc[0].image_id,'cosine_single_vs_batch':cosine})


## 7. Extração completa, com retomada

Ativar após piloto e revisão dos dados/protocolo. Uma extração por backbone contém todos os splits, cada um em sua pasta. Não é feita por fração/seed. Teste só passa pelo extrator congelado; nenhuma métrica de teste é usada aqui.

Cada bloco NPZ contém embeddings float32, image_id, class_id, split e identidade de features. O config associado registra modelo/revisão/normalização/dados/código/ambiente/GPU/batch. A identidade do chunk é conferida antes de reutilizá-lo, com checksum do arquivo e ordem dos IDs.

Para testar retomada, use MAX_NEW_CHUNKS=2, execute, depois None e execute novamente. Para outra sessão, preserve a pasta OUT e aponte PREVIOUS_OUTPUT_DIR para a saída anterior anexada; mantenha a configuração. Alterar batch/ambiente/GPU gera outra identidade por cautela numérica.

Volume bruto previsto: DINOv3 ~97,7 MB; ResNet ~195,4 MB para 23.850 vetores float32, além de metadados. Classificadores rodam em CPU.

In [10]:
# Manifesto revisado, sem alterações.
APPROVED_MANIFEST_SHA256 = (
    "80d6eedc883410b47beeee6b1c72e72d69309ba38ac2595fc8f2590e6928103d"
)

# Confere se o arquivo carregado é exatamente o revisado.
assert MANIFEST_HASH == APPROVED_MANIFEST_SHA256, (
    "O manifesto carregado é diferente do revisado. Confira PROTOCOL_DIR."
)

DATA_APPROVED = True
REVIEWERS = "Paula Eduarda de Lima"  # Inclua Ana Júlia se ela também revisou.

print("Manifesto revisado confirmado:", MANIFEST_HASH)

BUDGET_REVIEWED = True

require_main()
print("Manifesto e orçamento registrados. Extração liberada.")

Manifesto revisado confirmado: 80d6eedc883410b47beeee6b1c72e72d69309ba38ac2595fc8f2590e6928103d
Manifesto e orçamento registrados. Extração liberada.


In [11]:
FEATURE_DIR = None
if LOAD_BACKBONE:
    assert globals().get('MODEL_READY', False), 'Resolva o carregamento do backbone antes de preparar features.'
    FEATURE_CONFIG={'model':BACKBONE,'identity':adapter.identity,'preprocessing':adapter.preprocessing,
        'dimension':adapter.dim,'manifest_sha256':MANIFEST_HASH,'code_hash':CODE_HASH,
        'environment':ENV,'gpu':GPU,'inference_dtype':'float16','storage_dtype':'float32',
        'batch':EXTRACTION_BATCH,'chunk_size':CHUNK_SIZE}
    FEATURE_ID=digest(FEATURE_CONFIG)
    FEATURE_DIR=OUT/'features'/f'{BACKBONE}_{FEATURE_ID[:16]}'
if RUN_EXTRACTION:
    require_main();assert LOAD_BACKBONE
    result=extract_chunks(manifest,FEATURE_DIR,FEATURE_CONFIG,
        lambda rows:adapter.embed(rows,DATA_ROOT,EXTRACTION_BATCH),CHUNK_SIZE,MAX_NEW_CHUNKS)
    print(result)
print('Pasta de features:',FEATURE_DIR)


{'complete': True, 'new': 94, 'reused': 0, 'seconds': 1025.1531221239998, 'feature_id': '3ee57e90677e7f6934cdf6a856b9b4a8a814d4a44582779f81fb886ee46e79b5'}
Pasta de features: /kaggle/working/oct_models/80d6eedc8834_a305b2c7/features/resnet50_3ee57e90677e7f69


## 8. Rotina linear única para todos os backbones

Em uma sessão CPU, informe FEATURE_DIR_OVERRIDE com a pasta de features completa anexada. O adaptador de Ana Júlia deve exportar o mesmo contrato (config/feature_id/chunks/IDs); dimensão é lida do config, sem assumir1024.

Treina quatro frações × três seeds e 100% uma vez: **13 seleções por backbone, até 65 fits de C**. Quatro backbones: 52 seleções, até260 fits. “60 ajustes” era uma conta máxima sem eliminar repetições determinísticas e sem contar a busca de C.

Scaler é ajustado separadamente para cada subconjunto. A seleção recebe somente treino e validação; o split de teste nem é carregado. Empate favorece menor C. Salva classificador, trials, configuração, previsões e métricas de validação. O loader valida IDs e reordena embeddings usando as listas compartilhadas.

In [12]:
# from pathlib import Path

# RUN_EXTRACTION = False
# RUN_LINEAR = True
# RUN_LINEAR_FINAL_TEST = False

# FEATURE_DIR = Path(
#     "/kaggle/working/oct_models/80d6eedc8834_a305b2c7/"
#     "features/dinov3_vitl16_f101b16b77d8c857"
# )

# assert FEATURE_DIR.is_dir(), "Restaure o ZIP antes de continuar."
# assert (FEATURE_DIR / "config.json").is_file()
# require_main()

# print("Pronto para treinar usando:", FEATURE_DIR)

In [13]:
# LOAD_BACKBONE = False
# RUN_BENCHMARK = False
# RUN_EXTRACTION = False
# RUN_LINEAR = True

In [22]:
# RUN_EXTRACTION = False
# RUN_LINEAR = True
# RUN_LINEAR_FINAL_TEST = False

# assert FEATURE_DIR is not None, "Execute a seção 7 primeiro."

# config_features = json.loads(
#     (FEATURE_DIR / "config.json").read_text()
# )

# assert config_features["model"] == "resnet50", (
#     "A pasta selecionada não contém embeddings da ResNet-50."
# )
# assert config_features["manifest_sha256"] == MANIFEST_HASH

# require_main()

# print("Modelo:", config_features["model"])
# print("Dimensão:", config_features["dimension"])
# print("Pasta:", FEATURE_DIR)

Modelo: resnet50
Dimensão: 2048
Pasta: /kaggle/working/oct_models/80d6eedc8834_a305b2c7/features/resnet50_3ee57e90677e7f69


In [23]:
FEATURE_DIR_OVERRIDE = ''
if FEATURE_DIR_OVERRIDE:FEATURE_DIR=Path(FEATURE_DIR_OVERRIDE)
if RUN_LINEAR:
    require_main();assert FEATURE_DIR is not None
    fcfg=json.loads((FEATURE_DIR/'config.json').read_text())
    assert fcfg['manifest_sha256']==MANIFEST_HASH
    tr,Xtr=load_split(FEATURE_DIR,manifest,'train');va,Xva=load_split(FEATURE_DIR,manifest,'val')
    fid=digest(fcfg);linear_root=OUT/'runs'/f"{fcfg['model']}_{fid[:16]}_linear"
    selections=[]
    for pct in [1,5,10,25,100]:
        for seed in ([17] if pct==100 else [17,42,2026]):
            subset=lists[f'p{pct:02d}_seed{seed}']
            config={'feature_id':fid,'manifest_sha256':MANIFEST_HASH,'model':fcfg['model'],
             'percent':pct,'subset_seed':seed,'subset_ids_sha256':digest(sorted(subset.image_id)),
             'C_grid':LINEAR_C,'max_iter':LINEAR_MAX_ITER,'scaler_fit':'selected_train_only',
             'class_order':CLASSES,'code_hash':CODE_HASH,'environment':ENV,'selection':'val_macro_f1_smallest_C_tie'}
            run=linear_root/f'p{pct:02d}_seed{seed}_{digest(config)[:8]}'
            selected=fit_linear(tr,Xtr,va,Xva,subset,run,config)
            selections.append({'run':str(run.relative_to(OUT)),**selected})
    # Registry is frozen before any test evaluation. Existing mismatch is refused.
    lock=linear_root/'selection_lock.json'
    payload={'feature_id':fid,'manifest_sha256':MANIFEST_HASH,'selections':selections}
    if lock.exists():assert json.loads(lock.read_text())==payload,'Seleção anterior diferente; crie nova versão.'
    else:atomic_json(lock,payload)
    display(pd.DataFrame(selections))
    print('Registro da seleção:',lock)


,run,C,validation_macro_f1,classifier_sha256,config_hash,test_used_for_selection
0,runs/resnet50_3ee57e90677e7f69_linear/p01_seed...,0.010,0.710407,6d2c67dd28f9c96af99fab4842ff13f4a45c21a8c8eab1...,633e87762e7a9ccd844b45af3ebc1f5e9f40e0f4bf06c0...,False
1,runs/resnet50_3ee57e90677e7f69_linear/p01_seed...,0.100,0.714828,79cbe3cb9de11e824627a17916a7df7f05a87d585b28a7...,f08f68f4954659715ff62ed140d78ff9f8d823d5825bd0...,False
2,runs/resnet50_3ee57e90677e7f69_linear/p01_seed...,0.010,0.719690,28bb0d9b6be987862399b296743b7ec170a58a75c32bb8...,05455296680f6c5ae9c28332fd3d9d25b5229f668bf91e...,False
3,runs/resnet50_3ee57e90677e7f69_linear/p05_seed...,0.100,0.823944,aeef4b060e42db413f94d2699897a2b3f0d6e575609093...,d814a6815706d0f7951842f176ae4fab3c9f611768eed8...,False
4,runs/resnet50_3ee57e90677e7f69_linear/p05_seed...,0.010,0.808528,0b55f6dcd2ca2b5c02ce10d398b2f16dd05a0f3473deca...,de22b880a53dfa398537fc7f9671a54ae95bc4db0d83d3...,False
5,runs/resnet50_3ee57e90677e7f69_linear/p05_seed...,0.100,0.824940,73e1daa0b9e7b41168e4d47153f63bd2f3faf618bbec1b...,8da110b847eae619012a2115bc1b13f36b51347a7684be...,False
6,runs/resnet50_3ee57e90677e7f69_linear/p10_seed...,0.010,0.848933,e0492e2c9efffa0dc801b245d0a002f39cd524ed14ecbf...,f71e9c8b328aa1da6d52b4ddc8bef56ca51dd682418880...,False
7,runs/resnet50_3ee57e90677e7f69_linear/p10_seed...,0.010,0.844530,d9df4c8fcbc6888622bb12750cdc75639063b4e223be2b...,c1161c2c048ca52814a61a87eff12ac4b6df46c5070414...,False
8,runs/resnet50_3ee57e90677e7f69_linear/p10_seed...,0.100,0.852810,e6d79275df724b8730d9f6ed830a0ff4b7e2ad7b7663fd...,560f4cf8f64f32c4494bba8d1d40609ccf97deb56efe1c...,False
9,runs/resnet50_3ee57e90677e7f69_linear/p25_seed...,0.010,0.879743,3ab390f56a1c9f6ddb0ad9b65e231361d1ff0cea1fe50d...,68fcd30f0c370a5b54fa16029c6c1b4469cc4fe3179bee...,False


Registro da seleção: /kaggle/working/oct_models/80d6eedc8834_a305b2c7/runs/resnet50_3ee57e90677e7f69_linear/selection_lock.json


## 9. Avaliação final linear — opcional e separada

Só executar depois de encerrar as decisões e o orçamento de busca. A seção lê o registro de seleção e os classificadores salvos, sem refit nem escolha de C. Modificar configurações depois de observar teste exige declarar essa contaminação; um flag não impede decisões humanas indevidas.

Informe o selection_lock.json exibido acima. Joblib só deve carregar classificadores produzidos e preservados pelo projeto.

In [24]:
LINEAR_SELECTION_LOCK = ''
if RUN_LINEAR_FINAL_TEST:
    require_main();assert FEATURE_DIR is not None
    lock_path=Path(LINEAR_SELECTION_LOCK);selected=json.loads(lock_path.read_text())
    assert selected['manifest_sha256']==MANIFEST_HASH
    assert selected['feature_id']==digest(json.loads((FEATURE_DIR/'config.json').read_text()))
    te,Xte=load_split(FEATURE_DIR,manifest,'test')
    final=[]
    for entry in selected['selections']:
        run=OUT/entry['run'];assert sha(run/'classifier.joblib')==entry['classifier_sha256']
        assert digest(json.loads((run/'config.json').read_text()))==entry['config_hash']
        clf=joblib.load(run/'classifier.joblib');pred=predict_frame(clf,Xte,te.image_id)
        m=save_evaluation(run,te,pred,'test');final.append({'run':entry['run'],**{k:m[k] for k in ['macro_f1','accuracy','balanced_accuracy','auroc_macro']}})
    pd.DataFrame(final).to_csv(lock_path.parent/'test_summary.csv',index=False);display(pd.DataFrame(final))


## 10. Ajuste parcial DINOv3 — medir uma época antes

Ajusta apenas os **dois últimos blocos e uma nova cabeça linear**; embeddings, blocos anteriores e norm final permanecem congelados. O código verifica `model.layer`, estrutura da versão4.56.2. É fine-tuning parcial, não treinamento completo.

Batch2 × acumulação8 → efetivo16, exceto último grupo. AMP com GradScaler, gradientes normalizados pelo número real de exemplos e clipping1. Sem inversão vertical, crops ou augmentations. Normalização e geometria iguais à extração.

O piloto executa uma época real em10%, seed17, e validação inteira. Não usar seus pesos como inicialização das buscas: cada taxa/seed começa do checkpoint original. Se OOM, reduzir FT_BATCH e aumentar acumulação para preservar o batch efetivo; registrar em novo run.

Checkpoint `last.pt` contém parâmetros treináveis, otimizador, scaler, estados aleatórios, gerador de shuffle, melhor estado e contador de early stopping. Retoma **na fronteira de época**, repetindo a época interrompida; não promete retomar exatamente um minibatch. Os pesos congelados são reconstruídos da revisão fixada. Não reutilizar embeddings antigos durante ajuste.

GPU tem de estar disponível. Carregar checkpoints torch somente de saídas próprias confiáveis.

In [35]:
def new_dino():
    import torch
    from oct_gpu import Backbone
    from kaggle_secrets import UserSecretsClient
    return Backbone('dinov3_vitl16','cuda:0',CACHE,DINO_REVISION,UserSecretsClient().get_secret('HF_TOKEN'))

def ft_config(adapter,pct,seed,lr,epochs,purpose):
    return {'model':'dinov3_vitl16','identity':adapter.identity,'preprocessing':adapter.preprocessing,
       'manifest_sha256':MANIFEST_HASH,'code_hash':CODE_HASH,'environment':ENV,
       'gpu':torch.cuda.get_device_name(0),'percent':pct,'seed':seed,
       'subset_ids_sha256':digest(sorted(lists[f'p{pct:02d}_seed{seed}'].image_id)),
       'lr':lr,'epochs':epochs,'patience':FT_PATIENCE,'weight_decay':.01,
       'batch':FT_BATCH,'accumulation':FT_ACCUMULATION,'eval_batch':FT_EVAL_BATCH,
       'trainable':'last_two_blocks_and_head_only','purpose':purpose}

if RUN_FT_EPOCH_PILOT:
    import torch
    from oct_gpu import fit_partial
    assert torch.cuda.is_available()
    if 'adapter' in globals():del adapter;gc.collect();torch.cuda.empty_cache()
    ft_adapter=new_dino()
    cfg=ft_config(ft_adapter,10,17,FT_LRS[0],1,'epoch_timing_pilot')
    run=OUT/'ft_pilot'/digest(cfg)[:16]
    result=fit_partial(ft_adapter,lists['p10_seed17'],lists['val'],DATA_ROOT,run,cfg)
    history=json.loads((run/'history.json').read_text());h=history[0]
    epoch_seconds=h['train_seconds']+h['validation_seconds']+h.get('checkpoint_seconds',0)
    print('Segundos por época (inclui validação/salvamento):',epoch_seconds)
    print('Estimativa por configuração de 15 épocas +30%:',epoch_seconds*15*1.3/3600,'horas')
    print('Multiplique por taxas, seeds e frações; não assume early stopping.')
    del ft_adapter;gc.collect();torch.cuda.empty_cache()


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

{'epoch': 1, 'loss': 1.8598204495614035, 'val_macro_f1': 0.5651861122415458, 'train_seconds': 56.74060118599982, 'validation_seconds': 57.49365297099985, 'peak_allocated_gib': 1.5297045707702637, 'improved': True, 'checkpoint_seconds': 0.28182169599995177}
Segundos por época (inclui validação/salvamento): 114.51607585299962
Estimativa por configuração de 15 épocas +30%: 0.6202954108704146 horas
Multiplique por taxas, seeds e frações; não assume early stopping.


## 11. Busca de ajuste parcial e retomada

Começar 10%/seed17; depois completar seeds42/2026; só adicionar1% se houver orçamento. Cada modelo/fração/seed seleciona entre duas taxas pela validação. Com2 modelos×2 frações×3 seeds são12 configurações finais selecionadas, mas até24 treinamentos candidatos se usar duas taxas — além dos pilotos.

Cada execução salva melhor checkpoint, último checkpoint, parâmetros alterados, histórico, tempo, memória, previsões e métricas de validação. Registra se melhorou no limite de épocas. Resume somente com configuração idêntica e saídas restauradas. O notebook executa sequencialmente e pode demorar horas; selecione poucas combinações por sessão.

In [26]:
if RUN_PARTIAL_FT:
    require_main()
    import torch
    from oct_gpu import fit_partial
    assert torch.cuda.is_available()
    assert all(p in [1,10] for p in FT_PERCENTAGES) and all(s in [17,42,2026] for s in FT_SEEDS)
    assert 10<=FT_EPOCHS<=20 and len(FT_LRS)<=3
    if 'adapter' in globals():del adapter;gc.collect();torch.cuda.empty_cache()
    for pct in FT_PERCENTAGES:
        for seed in FT_SEEDS:
            choices=[]
            for lr in sorted(FT_LRS):
                ft_adapter=new_dino();cfg=ft_config(ft_adapter,pct,seed,lr,FT_EPOCHS,'main_partial_finetuning')
                run=OUT/'partial_ft'/f'p{pct:02d}_seed{seed}'/digest(cfg)[:16]
                result=fit_partial(ft_adapter,lists[f'p{pct:02d}_seed{seed}'],lists['val'],DATA_ROOT,run,cfg)
                choices.append({'lr':lr,'run':str(run.relative_to(OUT)),**result})
                del ft_adapter;gc.collect();torch.cuda.empty_cache()
            # Empate favorece menor lr.
            chosen=sorted(choices,key=lambda c:(-c['validation_macro_f1'],c['lr']))[0]
            lock=OUT/'partial_ft'/f'p{pct:02d}_seed{seed}'/'selection_lock.json'
            payload={'manifest_sha256':MANIFEST_HASH,'choices':choices,'selected':chosen}
            if lock.exists():assert json.loads(lock.read_text())==payload,'Registro de seleção já difere; revisar versão.'
            else:atomic_json(lock,payload)
            print('Selecionado:',chosen,'Registro:',lock)


## 12. Avaliação final do ajuste parcial — opcional

Informe um registro de seleção de fração/seed já encerrado. Recarrega checkpoint original + parâmetros ajustados escolhidos, avalia imagens de teste diretamente e salva previsões. Não usa os embeddings congelados.

In [27]:
FT_SELECTION_LOCK = ''
if RUN_FT_FINAL_TEST:
    require_main()
    import torch
    from oct_gpu import PartialDINO,restore_trainable,probabilities
    lock=json.loads(Path(FT_SELECTION_LOCK).read_text());assert lock['manifest_sha256']==MANIFEST_HASH
    run=OUT/lock['selected']['run'];assert sha(run/'best.pt')==lock['selected']['best_sha256']
    cfg=json.loads((run/'config.json').read_text())
    assert cfg['environment']==ENV and cfg['code_hash']==CODE_HASH
    if 'adapter' in globals():del adapter;gc.collect();torch.cuda.empty_cache()
    ft_adapter=new_dino();assert cfg['identity']==ft_adapter.identity and cfg['preprocessing']==ft_adapter.preprocessing
    net=PartialDINO(ft_adapter.model).to(ft_adapter.device)
    state=torch.load(run/'best.pt',map_location='cpu',weights_only=False);assert state['config']==cfg
    restore_trainable(net,state['trainable'])
    pred=probabilities(net,lists['test'],ft_adapter,DATA_ROOT,FT_EVAL_BATCH)
    metrics=save_evaluation(run,lists['test'],pred,'test');display(pd.Series(metrics))
    del net,ft_adapter;gc.collect();torch.cuda.empty_cache()


## 13. Backup, commit e conclusão

Execute esta célula ao fim de cada etapa, inclusive extração parcial. Verifica o ZIP por CRC; baixe/preserve antes de encerrar. Checkpoints podem ocupar bastante espaço; confirme a quota livre. Não adicionar embeddings, pesos, ZIP ou joblib ao Git. Preservar esses arquivos em saídas/dataset privado e registrar seus caminhos/checksums.

Commit: notebook sem saída Base64, `src/*.py`, protocolo, configurações sem credenciais, benchmarks, parâmetros treináveis, trials, métricas, históricos e registros de seleção. Guardar ambiente/manifesto/listas do notebook02 como referência. Não declarar resultados que não foram executados.

Para Ana Júlia: reutilizar `evaluate_predictions` e `fit_linear`. Exportar features no mesmo schema com sua própria identidade de checkpoint/pooling/dimensão/preprocessamento; não atribuir automaticamente a configuração DINOv3 ao RETFound/ViT. CNN é um experimento separado, sem pré-treino, com orçamento próprio revisado.

Concluído somente após: backbones carregados e piloto real; batch escolhido; extração completa conferida e recuperável; seleção linear na validação; checkpoint do ajuste retomado em sessão nova; comparação final depois de decisões encerradas. Limitações de pacientes e pré-treinamento continuam vigentes.

Fontes: [DINOv3 v4.56.2 — implementação](https://github.com/huggingface/transformers/blob/v4.56.2/src/transformers/models/dinov3_vit/modeling_dinov3_vit.py), [ResNet50](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html), [AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html), [AUROC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

In [36]:
MAKE_BACKUP = True
if MAKE_BACKUP:
    from zipfile import ZipFile,ZIP_DEFLATED
    entries={str(p.relative_to(OUT)):sha(p) for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='checksums.json' and p.suffix!='.tmp'}
    atomic_json(OUT/'checksums.json',entries)
    archive=OUT_ROOT/f'{OUT.name}_backup.zip'
    with ZipFile(archive,'w',ZIP_DEFLATED) as z:
        for p in OUT.rglob('*'):
            if p.is_file() and p.suffix!='.tmp':z.write(p,arcname=str(p.relative_to(OUT)))
        for p in SRC.glob('*.py'):z.write(p,arcname='src/'+p.name)
    with ZipFile(archive) as z:assert z.testzip() is None
    print('Backup:',archive,'GB:',archive.stat().st_size/1e9,'SHA-256:',sha(archive))
    print('Preserve a saída desta versão e baixe o ZIP. Retome com PREVIOUS_OUTPUT_DIR.')


Backup: /kaggle/working/oct_models/80d6eedc8834_a305b2c7_backup.zip GB: 0.552653574 SHA-256: d4da4e28cad56dab97a686c07accf0fd9f44c9f4324c629a19fea1af8800309e
Preserve a saída desta versão e baixe o ZIP. Retome com PREVIOUS_OUTPUT_DIR.


In [37]:
from pathlib import Path
from IPython.display import HTML, display
import base64

arquivo = Path(
    "/kaggle/working/oct_models/"
    "80d6eedc8834_a305b2c7_backup.zip"
)

assert arquivo.is_file(), (
    "Execute primeiro a célula de backup com MAKE_BACKUP=True."
)

tamanho_mb = arquivo.stat().st_size / 1024**2
print(f"Tamanho: {tamanho_mb:.2f} MB")

if tamanho_mb <= 15:
    conteudo = base64.b64encode(arquivo.read_bytes()).decode("ascii")
    display(HTML(
        f'<a download="{arquivo.name}" '
        f'href="data:application/zip;base64,{conteudo}">'
        'Baixar resultados do piloto DINOv3</a>'
    ))
else:
    print("Arquivo grande: preserve e baixe pelas saídas do notebook.")

Tamanho: 527.05 MB
Arquivo grande: preserve e baixe pelas saídas do notebook.
